In [2]:
from keras.layers import Bidirectional, Concatenate, Permute, Dot, Input, LSTM, Multiply, RepeatVector, Dense, Activation, Lambda
from keras.optimizers import Adam
from keras.utils import to_categorical
from keras.models import load_model, Model
import keras.backend as K
import tensorflow as tf
import numpy as np

from faker import Faker
import random
from tqdm import tqdm
from babel.dates import format_date
from nmt_utils import *
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
m = 10000
dataset, human_vocab, machine_vocab, inv_machine_vocab = load_dataset(m)

100%|██████████| 10000/10000 [00:01<00:00, 5923.97it/s]


In [4]:
dataset[:10]

[('4 sep 1993', '1993-09-04'),
 ('31.07.70', '1970-07-31'),
 ('11/24/16', '2016-11-24'),
 ('monday december 22 1986', '1986-12-22'),
 ('sunday december 9 1990', '1990-12-09'),
 ('friday january 2 1981', '1981-01-02'),
 ('saturday march 2 2002', '2002-03-02'),
 ('11 mar 1979', '1979-03-11'),
 ('23 jan 1977', '1977-01-23'),
 ('tuesday august 9 1994', '1994-08-09')]

In [5]:
human_vocab

{' ': 0,
 '.': 1,
 '/': 2,
 '0': 3,
 '1': 4,
 '2': 5,
 '3': 6,
 '4': 7,
 '5': 8,
 '6': 9,
 '7': 10,
 '8': 11,
 '9': 12,
 'a': 13,
 'b': 14,
 'c': 15,
 'd': 16,
 'e': 17,
 'f': 18,
 'g': 19,
 'h': 20,
 'i': 21,
 'j': 22,
 'l': 23,
 'm': 24,
 'n': 25,
 'o': 26,
 'p': 27,
 'r': 28,
 's': 29,
 't': 30,
 'u': 31,
 'v': 32,
 'w': 33,
 'y': 34,
 '<unk>': 35,
 '<pad>': 36}

In [6]:
tf.config.list_physical_devices()

E0000 00:00:1793442655.720599    6434 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1793442655.721755   11204 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1793442655.769732    6434 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [7]:
Tx = 30
Ty = 10
X, Y, Xoh, Yoh = preprocess_data(dataset, human_vocab, machine_vocab, Tx, Ty)

print("X.shape:", X.shape)
print("Y.shape:", Y.shape)
print("Xoh.shape:", Xoh.shape)
print("Yoh.shape:", Yoh.shape)

X.shape: (10000, 30)
Y.shape: (10000, 10)
Xoh.shape: (10000, 30, 37)
Yoh.shape: (10000, 10, 11)


In [8]:
index = 0
print("Source date:", dataset[index][0])
print("Target date:", dataset[index][1])
print()
print("Source after preprocessing (indices):", X[index])
print("Target after preprocessing (indices):", Y[index])
print()
print("Source after preprocessing (one-hot):", Xoh[index])
print("Target after preprocessing (one-hot):", Yoh[index])

Source date: 4 sep 1993
Target date: 1993-09-04

Source after preprocessing (indices): [ 7  0 29 17 27  0  4 12 12  6 36 36 36 36 36 36 36 36 36 36 36 36 36 36
 36 36 36 36 36 36]
Target after preprocessing (indices): [ 2 10 10  4  0  1 10  0  1  5]

Source after preprocessing (one-hot): [[0. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]
 [0. 0. 0. ... 0. 0. 1.]]
Target after preprocessing (one-hot): [[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]]


# Neural Machine Translation with Attention

In [9]:
repeator = RepeatVector(Tx)
concatenator = Concatenate(axis=-1)
densor1 = Dense(10, activation="tanh")
densor2 = Dense(1, activation="relu")
activator = Activation(softmax, name='attention_weights')
dotor = Dot(axes=1)

In [10]:
def one_step_attention(a, s_prev):
    """
    Performs one step of attention: Outputs a context vector computed as a dot product of the attention weights
    "alphas" and the hidden states "a" of the Bi-LSTM.
    
    Arguments:
    a -- hidden state output of the Bi-LSTM, numpy-array of shape (m, Tx, 2*n_a)
    s_prev -- previous hidden state of the (post-attention) LSTM, numpy-array of shape (m, n_s)
    
    Returns:
    context -- context vector, input of the next (post-attention) LSTM cell
    """

    s_prev = repeator(s_prev)
    concat = concatenator([a, s_prev])
    e = densor1(concat)
    energies = densor2(e)
    alphas = activator(energies)
    context = dotor([alphas, a])

    return context

In [11]:
n_a = 32    # pre_attention units number
n_s = 64    # post_attention units number

post_attention_LSTM_cell = LSTM(n_s, return_state=True)
output_layer = Dense(len(machine_vocab), activation=softmax)

In [26]:
def modelf(Tx, Ty, n_a, n_s, human_vocab_size, machine_vocab_size):
    """
    Arguments:
    Tx -- length of the input sequence
    Ty -- length of the output sequence
    n_a -- hidden state size of the Bi-LSTM
    n_s -- hidden state size of the post-attention LSTM
    human_vocab_size -- size of the python dictionary "human_vocab"
    machine_vocab_size -- size of the python dictionary "machine_vocab"

    Returns:
    model -- Keras model instance
    """

    X = Input(shape=(Tx, human_vocab_size))
    s0 = Input(shape=(n_s,), name='s0')
    c0 = Input(shape=(n_s,), name='c0')
    s = s0
    c = c0

    outputs = []

    a = Bidirectional(LSTM(units=n_a, return_sequences=True))(X)

    for t in range(Ty):

        context = one_step_attention(a, s)
        # context = RepeatVector(1)(context)
        s, _, c = post_attention_LSTM_cell(context, initial_state=[s, c])

        out = output_layer(s)
        outputs.append(out)

    model = Model(inputs=[X, s0, c0], outputs=outputs)

    return model

In [27]:
model = modelf(Tx, Ty, n_a, n_s, len(human_vocab), len(machine_vocab))

In [28]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 30, 37)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s0 (InputLayer)     │ (None, 64)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_6     │ (None, 30, 64)    │     17,920 │ input_layer_6[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 30, 64)    │          0 │ s0[0][0],         │
│ (RepeatVector)      │                   │            │ lstm[12][0],      │
│                     │                   │            │ lstm[13][0],      │
│                     │                   │            │ lstm[14][0],      │
│                     │                   │            │ lstm[15][0],      │
│                     │                   │            │ lstm[16][0],      │
│                     │                   │            │ lstm[17][0],      │
│                     │                   │            │ lstm[18][0],      │
│                     │                   │            │ lstm[19][0],      │
│                     │                   │            │ lstm[20][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 30, 128)   │          0 │ bidirectional_6[… │
│ (Concatenate)       │                   │            │ repeat_vector[17… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[18… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[19… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[20… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[21… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[22… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[23… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[24… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[25… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[26… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30, 10)    │      1,290 │ concatenate[17][… │
│                     │                   │            │ concatenate[18][… │
│                     │                   │            │ concatenate[19][… │
│                     │                   │            │ concatenate[20][… │
│                     │                   │            │ concatenate[21][… │
│                     │                   │            │ concatenate[22][… │
│                     │                   │            │ concatenate[23][… │
│                     │                   │            │ concatenate[24][… │
│                     │                   │            │ concatenate[25][

 Total params: 52,960 (206.88 KB)

 Trainable params: 52,960 (206.88 KB)

 Non-trainable params: 0 (0.00 B)

In [35]:
optimizer = Adam(learning_rate=0.005, beta_1=0.9, beta_2=0.999, weight_decay=0.01)
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=[['accuracy']]*10)

In [36]:
s0 = np.zeros((m, n_s))
c0 = np.zeros((m, n_s))
outputs = list(Yoh.swapaxes(0, 1))

In [37]:
model.fit([Xoh, s0, c0], outputs, epochs=1, batch_size=100)

100/100 ━━━━━━━━━━━━━━━━━━━━ 45s 107ms/step - dense_2_accuracy: 0.6176 - dense_2_accuracy_1: 0.8101 - dense_2_accuracy_2: 0.3067 - dense_2_accuracy_3: 0.0908 - dense_2_accuracy_4: 0.9556 - dense_2_accuracy_5: 0.3519 - dense_2_accuracy_6: 0.0648 - dense_2_accuracy_7: 0.9329 - dense_2_accuracy_8: 0.2175 - dense_2_accuracy_9: 0.0930 - dense_2_loss: 2.5689 - loss: 15.9972


In [38]:
model.load_weights('models/model.h5')

In [40]:
EXAMPLES = ['3 May 1979', '5 April 09', '21th of August 2016', 'Tue 10 Jul 2007', 
            'Saturday May 9 2018', 'March 3 2001', 'March 3rd 2001', '1 March 2001']

s00 = np.zeros((1, n_s))
c00 = np.zeros((1, n_s))

for example in EXAMPLES:
    # 1. Convert string → integers
    source = string_to_int(example, Tx, human_vocab)
    
    # 2. One-hot encode and reshape to (1, Tx, human_vocab_size)
    source = np.array(
        list(map(lambda x: to_categorical(x, num_classes=len(human_vocab)), source))
    )
    source = np.expand_dims(source, axis=0)          # shape: (1, Tx, vocab_size)
    
    # 3. Predict (returns a list of Ty outputs)
    prediction = model.predict([source, s00, c00], verbose=0)
    
    # 4. Convert list of predictions → array and take argmax
    prediction = np.argmax(np.array(prediction), axis=-1)   # shape: (Ty, 1)
    
    # 5. Map indices back to characters
    output = [inv_machine_vocab[int(idx)] for idx in prediction[:, 0]]
    
    print("source:", example)
    print("output:", ''.join(output))
    print()

source: 3 May 1979
output: 1979-05-33

source: 5 April 09
output: 2009-04-05

source: 21th of August 2016
output: 2016-08-20

source: Tue 10 Jul 2007
output: 2007-07-10

source: Saturday May 9 2018
output: 2018-05-09

source: March 3 2001
output: 2001-03-03

source: March 3rd 2001
output: 2001-03-03

source: 1 March 2001
output: 2001-03-01



In [42]:
def translate_date(sentence):
    # 1. Convert string → integers
    source = string_to_int(sentence, Tx, human_vocab)
    
    # 2. One-hot encode and reshape to (1, Tx, human_vocab_size)
    source = np.array(
        list(map(lambda x: to_categorical(x, num_classes=len(human_vocab)), source))
    )
    source = np.expand_dims(source, axis=0)          # (1, Tx, vocab_size)
    
    # 3. Predict (model returns a list of Ty outputs)
    prediction = model.predict([source, s00, c00], verbose=0)
    
    # 4. Convert list → array and take argmax
    prediction = np.argmax(np.array(prediction), axis=-1)   # (Ty, 1)
    
    # 5. Map indices to characters
    output = [inv_machine_vocab[int(idx)] for idx in prediction[:, 0]]
    
    print("source:", sentence)
    print("output:", ''.join(output))
    print()

# Test
example = "4th of july 2001"
translate_date(example)

source: 4th of july 2001
output: 2001-07-04



# Visualizing Attention

In [43]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 30, 37)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ s0 (InputLayer)     │ (None, 64)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_6     │ (None, 30, 64)    │     17,920 │ input_layer_6[0]… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector       │ (None, 30, 64)    │          0 │ s0[0][0],         │
│ (RepeatVector)      │                   │            │ lstm[12][0],      │
│                     │                   │            │ lstm[13][0],      │
│                     │                   │            │ lstm[14][0],      │
│                     │                   │            │ lstm[15][0],      │
│                     │                   │            │ lstm[16][0],      │
│                     │                   │            │ lstm[17][0],      │
│                     │                   │            │ lstm[18][0],      │
│                     │                   │            │ lstm[19][0],      │
│                     │                   │            │ lstm[20][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 30, 128)   │          0 │ bidirectional_6[… │
│ (Concatenate)       │                   │            │ repeat_vector[17… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[18… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[19… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[20… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[21… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[22… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[23… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[24… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[25… │
│                     │                   │            │ bidirectional_6[… │
│                     │                   │            │ repeat_vector[26… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 30, 10)    │      1,290 │ concatenate[17][… │
│                     │                   │            │ concatenate[18][… │
│                     │                   │            │ concatenate[19][… │
│                     │                   │            │ concatenate[20][… │
│                     │                   │            │ concatenate[21][… │
│                     │                   │            │ concatenate[22][… │
│                     │                   │            │ concatenate[23][… │
│                     │                   │            │ concatenate[24][… │
│                     │                   │            │ concatenate[25][

 Total params: 158,882 (620.64 KB)

 Trainable params: 52,960 (206.88 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 105,922 (413.76 KB)

In [44]:
attention_map = plot_attention_map(model, human_vocab, inv_machine_vocab, "Tuesday 09 Oct 1993", num=7, n_s=64)

ValueError: setting an array element with a sequence.